In [1]:
import gymnasium as gym
import collections
import wandb
import numpy as np

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=secret_value_0)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bhalaniakshat (bhalaniakshat-dwarkadas-j-sanghvi-college-of-engineering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
ENV_NAME = "FrozenLake-v1"
GAMMA = 0.9
TEST_EPISODES = 20
RANDOM_STEPS = 100

In [4]:
class Agent:
    def __init__(self):
        self.env = gym.make(ENV_NAME)
        self.state, _ = self.env.reset()
        self.rewards = collections.defaultdict(float)
        self.transits = collections.defaultdict(collections.Counter)
        self.values = collections.defaultdict(float)
    def play_n_random_steps(self, count):
        for _ in range(count):
            action = self.env.action_space.sample()
            new_state, reward, terminated, truncated, _ = self.env.step(action)
            self.rewards[(self.state, action, new_state)] = reward
            self.transits[(self.state, action)][new_state] += 1
            
            if terminated or truncated:
                self.state, _ = self.env.reset()
            else:
                self.state = new_state
    def select_action(self, state):
        best_action, best_value = None, None
        for action in range(self.env.action_space.n):
            action_value = self.values[(state, action)]
            if best_value is None or best_value < action_value:
                best_value = action_value
                best_action = action
        return best_action
    def play_episode(self, env):
        total_reward = 0.0
        state, _ = env.reset()
        while True:
            action = self.select_action(state)
            new_state, reward, terminated, truncated, _ = env.step(action)
            
            # Update history even during evaluation
            self.rewards[(state, action, new_state)] = reward
            self.transits[(state, action)][new_state] += 1
            
            total_reward += reward
            if terminated or truncated:
                break
            state = new_state
        return total_reward
    def value_iteration(self):
        for state in range(self.env.observation_space.n):
            for action in range(self.env.action_space.n):
                action_value = 0.0
                target_counts = self.transits[(state, action)]
                total = sum(target_counts.values())
                
                if total == 0:
                    continue

                for tgt_state, count in target_counts.items():
                    reward = self.rewards[(state, action, tgt_state)]
                    best_action = self.select_action(tgt_state)
                    # Bellman Equation for Q-Values
                    action_value += (count / total) * (reward + GAMMA * self.values[(tgt_state, best_action)])
                
                self.values[(state, action)] = action_value

In [5]:
wandb.init(
        project="frozenlake-q-iteration",
        config={
            "env": ENV_NAME,
            "gamma": GAMMA,
            "test_episodes": TEST_EPISODES,
            "random_steps": RANDOM_STEPS
        }
    )

wandb: setting up run vk194hbe
wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260319_113514-vk194hbe
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run exalted-totem-2
wandb: ⭐️ View project at https://wandb.ai/bhalaniakshat-dwarkadas-j-sanghvi-college-of-engineering/frozenlake-q-iteration
wandb: 🚀 View run at https://wandb.ai/bhalaniakshat-dwarkadas-j-sanghvi-college-of-engineering/frozenlake-q-iteration/runs/vk194hbe


In [6]:
test_env = gym.make(ENV_NAME)
agent = Agent()
iter_no = 0
best_reward = 0.0

In [7]:
while True:
        iter_no += 1
        agent.play_n_random_steps(RANDOM_STEPS)
        agent.value_iteration()

        reward = 0.0
        for _ in range(TEST_EPISODES):
            reward += agent.play_episode(test_env)
        reward /= TEST_EPISODES
        
        # Log to WandB
        wandb.log({"reward": reward, "iteration": iter_no})
        
        if reward > best_reward:
            print(f"Iter {iter_no}: Best reward updated {best_reward:.3f} -> {reward:.3f}")
            best_reward = reward
            
        if reward > 0.80:
            print(f"Solved in {iter_no} iterations!")
            break
            
wandb.finish()

Iter 8: Best reward updated 0.000 -> 0.400
Iter 10: Best reward updated 0.400 -> 0.550
Iter 12: Best reward updated 0.550 -> 0.600
Iter 16: Best reward updated 0.600 -> 0.800
Iter 18: Best reward updated 0.800 -> 0.850
Solved in 18 iterations!


wandb: updating run metadata; uploading data; uploading requirements.txt
wandb: uploading data
wandb: uploading history steps 0-17, summary, console lines 0-5
wandb: 
wandb: Run history:
wandb: iteration ▁▁▂▂▃▃▃▄▄▅▅▆▆▆▇▇██
wandb:    reward ▁▁▁▁▁▁▁▄▂▆▅▆▆▆▆█▆█
wandb: 
wandb: Run summary:
wandb: iteration 18
wandb:    reward 0.85
wandb: 
wandb: 🚀 View run exalted-totem-2 at: https://wandb.ai/bhalaniakshat-dwarkadas-j-sanghvi-college-of-engineering/frozenlake-q-iteration/runs/vk194hbe
wandb: ⭐️ View project at: https://wandb.ai/bhalaniakshat-dwarkadas-j-sanghvi-college-of-engineering/frozenlake-q-iteration
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260319_113514-vk194hbe/logs
